# Manage Feedback

#### Imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# Add the parent directory (or another appropriate path) to sys.path so Python can find Exclusion_functions
notebook_dir = os.path.dirname(os.path.abspath('04_Optimization.ipynb'))
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..', '..', '..'))
function_dir = os.path.abspath(os.path.join(parent_dir, 'Function_Files'))
if function_dir not in sys.path:
    sys.path.append(function_dir)

import pandas as pd
from openai_api import OpenAIAgent
import nest_asyncio
nest_asyncio.apply()
from IPython.display import Markdown, display
def printmd(string):
    display(Markdown(string))
    
import Feedback_functions as ff
import Matching_functions as mf
import datetime

#### Variables

In [3]:
CATEGORY = "Cups"
today = datetime.datetime.now().strftime('%m.%d.%Y')
ip_path = f"{parent_dir}\\Data\\"
ip_path_cat = f"{parent_dir}\\Data\\{CATEGORY}\\"
OP_PATH = f"{parent_dir}\\Data\\{CATEGORY}\\Output\\"

In [4]:
im_final = pd.read_csv(f"{OP_PATH}{CATEGORY}_matches.csv")

#### Read in Feedback

In [5]:
im_final_with_feedback = ff.consolidate_feedback_from_excel([f"{ip_path_cat}Cups_Subs_with_Feedback.xlsx"], im_final)

--- Starting Consolidation Process for 1 Excel Files ---

[1/1] Processing file: Cups_Subs_with_Feedback.xlsx
  [INFO] Found 21 sheets.
    - Skipping ignored sheet: 'Summary'
    - [INFO] Blank row found. Concluding substitute processing for '1--VG16CF'.
    - [SUCCESS] Processed sheet '1--VG16CF', found 3 substitutes for target '1--VG16CF'.
    - [INFO] Blank row found. Concluding substitute processing for '1--F98HCF'.
    - [SUCCESS] Processed sheet '1--F98HCF', found 1 substitutes for target '1--F98HCF'.
    - [INFO] Blank row found. Concluding substitute processing for '1--VBCLHP12W'.
    - [SUCCESS] Processed sheet '1--VBCLHP12W', found 9 substitutes for target '1--VBCLHP12W'.
    - [INFO] Blank row found. Concluding substitute processing for '1--VG12CF'.
    - [SUCCESS] Processed sheet '1--VG12CF', found 1 substitutes for target '1--VG12CF'.
    - [INFO] Blank row found. Concluding substitute processing for '1--VBCLLH16DW'.
    - [SUCCESS] Processed sheet '1--VBCLLH16DW', found 

In [6]:
im_final_with_feedback

,Targets,Subs,Correct,Feedback,Subcategory,Target_embeddings,Target_for_embeddings,Sub_embeddings,Sub_for_embeddings
0,1--VG16CF,1--KC16S,Accept,NaN,None,"[-0.04062986746430397, 0.016798632219433784, -...",Cup Cup 1000/Case Clear Polyethylene Terephtha...,"[-0.03740070015192032, 0.016286110505461693, -...",Cup Cup 1000/Case Clear Polyethylene Terephtha...
1,1--VG16CF,1--KC16CP,Reject,KC16CP is a combo pack,None,"[-0.04062986746430397, 0.016798632219433784, -...",Cup Cup 1000/Case Clear Polyethylene Terephtha...,"[-0.04234791547060013, 0.016776390373706818, -...",Cup Cup 600/Case Clear Polyethylene Terephthal...
2,1--VG16CF,1--TP16,Reject,Dart,None,"[-0.04062986746430397, 0.016798632219433784, -...",Cup Cup 1000/Case Clear Polyethylene Terephtha...,"[-0.03746701031923294, 0.015023769810795784, -...",Cup Cup 1000/Case Clear Polyethylene Terephtha...
3,1--F98HCF,1--VBFLSS98,Accept,NaN,None,"[-0.028009280562400818, -0.005703808274120092,...",Lid Lid 1000/Case Plastic Disposable Cold Onl...,"[-0.027993761003017426, 0.014698099344968796, ...",Lid Lid 1000/Case Plastic Disposable 16-24...
4,1--VBCLHP12W,0--310114,Accept,NaN,None,"[-0.016626210883259773, -0.01373258512467146, ...",Cup Cup 1000/Case White Paper Hot Only 12 O...,NaN,NaN
...,...,...,...,...,...,...,...,...,...
71,1--VBCLH8W,0--VB-SMR-0080,Accept,NaN,None,"[-0.01945924386382103, -0.03220439329743385, -...",Cup Cup White Paper Hot Only 8 OZ 8 Ounce ...,NaN,NaN
72,1--VBCLH8W,0--VBCLHP8W,Accept,NaN,None,"[-0.01945924386382103, -0.03220439329743385, -...",Cup Cup White Paper Hot Only 8 OZ 8 Ounce ...,NaN,NaN
73,1--VBCLH8W,0--IP-SMR8,Accept,NaN,None,"[-0.01945924386382103, -0.03220439329743385, -...",Cup Cup White Paper Hot Only 8 OZ 8 Ounce ...,NaN,NaN
74,1--VBCLH8W,0--260212,Accept,NaN,None,"[-0.01945924386382103, -0.03220439329743385, -...",Cup Cup White Paper Hot Only 8 OZ 8 Ounce ...,NaN,NaN


#### Evaluate Results

In [7]:
ff.get_feedback_summary(im_final_with_feedback)

Correct,Accept,Reject,Consider,Accept Rate,Accept/Consider Rate
Count,47,28,1,61.84%,63.16%


In [8]:
ff.get_reviewed_coverage(im_final, im_final_with_feedback)

,Reviewed PL SKUs,Private Label,Total,% Reviewed PL SKUs,% Reviewed of Total
Qty,412527.17,730050.25,1680437.83,56.51%,24.55%
Gross Cost,11117464.00,20233998.00,59143780.00,54.94%,18.80%
Net Cost,10197643.00,18709125.00,55009698.00,54.51%,18.54%


In [9]:
im_final_with_feedback['Correct'] = im_final_with_feedback['Correct'].replace({'Accept': 'Match'})
im_final_with_feedback['Correct'] = im_final_with_feedback['Correct'].replace({'Consider': 'Match'})
im_final_with_feedback['Correct'] = im_final_with_feedback['Correct'].replace({'Reject': 'No Match'})

#### Get New Prompts/Matches

In [10]:
user_prompts, numerical_id_to_entity_id_map = ff.user_prompts_with_rules(im_final, im_final_with_feedback, use_subcategory_rules = False)

Starting user_prompts_with_rules function with unified rule generation...
Generating one unified set of rules from all feedback examples...


Running cost $0.0000: 100%|██████████| 1/1 [00:00<00:00,  1.44chunk/s]


Unified rule generation complete.


Generating Prompts: 100%|██████████| 4481/4481 [00:13<00:00, 342.52it/s]

Generated 4481 prompts.


In [11]:
for user_prompt in user_prompts:
    print(user_prompt)
    print('----'*40)
    break



Original Item:
Entity ID: 1
- Description: Cup Foam 20 Oz Tall Coca Cola Stock Print
- Beverage Cup Type: Cup
- Product Type Collapse: Cup
- Pack Size: 500/Case
- Color: White|Red
- Material: Polystyrene Foam
- Foodservice Global Attributes: Disposable
- Usage Temperature: Cold & Hot
- Beverage Cup Style: Insulated
- Product Capacity: 20 OZ
- Capacity Value 1: 20
- Capacity Preferred Metric 1: Ounce (OZ)
- Bottom Diameter (IN): 2.4
- Top Diameter (IN): 3.7
- Product Dimension Type: Tapered & Graduated Product Dimensions
- Wall Height (IN): 6.1
- Product Dimensions: 3.7X6.1X2.4 IN.
- Case Pack: 500

Top Similar Items from Descriptions:
Entity ID:  751
 - Description: Cup Foam 20 Oz Cafe G Stock Print 3.6 In Dia- Beverage Cup Type: Cup
- Product Type Collapse: Cup
- Pack Size: 500/Case
- Color: Red
- Material: Polystyrene Foam
- Foodservice Global Attributes: Disposable
- Usage Temperature: Cold & Hot
- Beverage Cup Style: Insulated
- Product Capacity: 20 OZ
- Capacity Value 1: 20
- Cap

In [12]:
im_final_test = mf.generate_matches(im_final, user_prompts,numerical_id_to_entity_id_map, hard_rules = "If mentioned, size/volume should be a match if not very close. Same with color, shape, etc. You should not be trying to swap a 20 oz product with a 16 oz product.", batch_model= False )

Initializing OpenAI agent with model: gpt-4o
Getting responses from API (batch=False)...


Running cost $24.6460: 100%|██████████| 141/141 [57:10<00:00, 24.33s/chunk]


Received 4481 responses
Responses: ['{\n  "Entity ID": 1,\n  "Matches": [737, 3468],\n  "reasoning": "The original item is a 20 oz foam cup with Coca Cola stock print, and the key attributes include material (Polystyrene Foam), color (White|Red), capacity (20 oz), and dimensions (3.7X6.1X2.4 IN). Among the similar items, Entity ID 737 and Entity ID 3468 match closely in terms of capacity, material, dimensions, and usage temperature. While Entity ID 3468 has a different color (Blue), it is still swappable as the other attributes align well. Entity ID 751 has a slightly different top diameter (3.6 IN) and wall height (6.8 IN), making it less ideal. Entity IDs 860, 863, 1114, and 869 have different capacities (24 oz or 32 oz), which disqualifies them. Entity IDs 749, 655, and 652 have either different capacities (16 oz) or usage temperature (Hot Only for Entity ID 749), making them unsuitable swaps."\n}', '{\n  "Entity ID": 2,\n  "Matches": [867, 3444],\n  "reasoning": "The items with Ent

#### Adjust New Matches From Feedback

In [13]:
im_final2 = ff.update_matches_based_on_feedback_sym(im_final_test, im_final_with_feedback, feedback_correct_col = 'Correct', accept_value = "Match", reject_value='No Match')

Starting update with strict target-centric rules...
Identified 20 unique Targets in feedback.
Processing 76 feedback entries to build allowed pairs...
Applying conditional filtering (Pass 1 of 2)...
Reconciling matches for global symmetry (Pass 2 of 2)...
Finished update with strict target-centric rules.


#### Write new matches

In [14]:
mf.write_top_matches(im_final2, f"{OP_PATH}{CATEGORY}_Subs_{today}.xlsx", n = 35, pl = True, enable_vpn_exclusion = True)

Starting write_top_matches with enable_vpn_exclusion=True
Looking for qty column: 'Qty'
Available columns: ['Entity--Item', 'Item Desc 1', 'Item Desc 2', 'Qty', 'Gross Cost', 'Net Cost', 'po_cost_amt', 'VB Flag', 'VGN', 'VPN', 'Pack Size', 'Combined Descriptions', 'Description with Attributes', 'attributes', 'Beverage Cup Type', 'Product Type Collapse', 'Color', 'Material', 'Foodservice Global Attributes', 'Usage Temperature', 'Beverage Cup Style', 'Sustainable Products', 'Product Capacity', 'Capacity Value 1', 'Capacity Preferred Metric 1', 'Lid, Cover & Cap Type', 'Lid, Cover & Cap Style', 'Compatible Product & Product Type', 'Bottom Diameter (IN)', 'Top Diameter (IN)', 'Product Dimension Type', 'Wall Height (IN)', 'Product Dimensions', 'Case Pack', 'for_embedding', 'embeddings', 'Matches', 'reasoning', 'description_display_col']
Found qty column 'Qty' with 4481 rows
Qty column is numeric, sorting by Qty
PL filter applied: 440 rows remain
Selected top 35 items
Processed target item: 

#### Save files

In [15]:
im_final2.to_csv(f"{OP_PATH}{CATEGORY}_matches_post_feedback_1.csv", index=False)
im_final_with_feedback.to_csv(f"{OP_PATH}{CATEGORY}_feedback_df.csv", index=False)